# SciGeoGuard-X — Final Merge and Freeze v9.1

## Final merge only: no new scientific experiments

This notebook merges the two final artifacts:

1. **GitHub-ready repository ZIP**  
   `SciGeoGuard-X_GitHub_Ready_All_Code_and_Results.zip`

2. **successful v9 closure ZIP**  
   `SciGeoGuard_X_v9_Final_Publication_Closure.zip`

It then creates the **single frozen evidence package** for the FCS *Code & Data in Earth Science* manuscript.

### What v9.1 fixes

The v9 closure run completed IMERG v8.2 successfully, but it could not see the earlier v3/v6/v8.1 files in that runtime. Therefore its `science_ready_for_fcs=false` flag and "missing result" figure panels were artifacts of incomplete input discovery.

v9.1 recomputes readiness only after it has all evidence in one workspace.

### No new model experiments

This notebook does **not** rerun:

- v3 benchmark generation;
- v5/v5.2 LLM judging;
- v6 7B models;
- Sentinel-2;
- WorldCover;
- CopDEM.

It only merges, validates, summarizes, plots, audits, freezes, and exports.

## 0. Runtime configuration

Normal CPU Colab is enough.

If you are using Colab, set `OPEN_UPLOAD_DIALOG=True` and upload the two ZIP files.
If the files are already in the runtime, leave it `False`.

The notebook rejects offline/dry-run fixtures as publication evidence.

In [ ]:
OPEN_UPLOAD_DIALOG = False
DRY_RUN_FIXTURES = False
SEED = 20261004

from pathlib import Path
from datetime import datetime, timezone
import os, io, re, json, math, random, shutil, zipfile, hashlib, tempfile
import numpy as np
import pandas as pd

random.seed(SEED)
np.random.seed(SEED)

ROOT = Path("/content") if Path("/content").exists() else Path("/mnt/data")
WORK = ROOT / "scigeoguard_v9_1"
INPUTS = WORK / "inputs"
EXTRACTED = WORK / "extracted"
OUTPUTS = WORK / "outputs"
FROZEN = OUTPUTS / "frozen_evidence"

for d in [WORK, INPUTS, EXTRACTED, OUTPUTS, FROZEN]:
    d.mkdir(parents=True, exist_ok=True)

print("SciGeoGuard-X v9.1")
print("ROOT =", ROOT)
print("WORK =", WORK)

## 1. Upload the two final archives

In [ ]:
def upload_files():
    if not Path("/content").exists():
        return []

    from google.colab import files
    uploaded = files.upload()
    paths = []

    for name, data in uploaded.items():
        p = INPUTS / name
        p.write_bytes(data)
        paths.append(p)

    return paths

if OPEN_UPLOAD_DIALOG:
    uploaded = upload_files()
    print("Uploaded:", [p.name for p in uploaded])
else:
    print("Upload dialog disabled; scanning existing runtime files.")

## 2. Dry-run fixture generator

The distributed notebook defaults to real inputs.  
This fixture mode exists only to test the complete merge pipeline without external files.

In [ ]:
def make_dryrun_archives():
    fixture_root = WORK / "fixture_source"
    repo = fixture_root / "SciGeoGuard-X"
    closure = fixture_root / "closure"

    if fixture_root.exists():
        shutil.rmtree(fixture_root)

    repo.mkdir(parents=True)
    closure.mkdir(parents=True)

    # ----- v3 -----
    v3dir = repo / "results/v3_controlled_and_real_data"
    v3dir.mkdir(parents=True)

    labels = (
        ["PROVEN"] * 145 +
        ["REFUTED"] * 278 +
        ["UNPROVEN"] * 22
    )

    pd.DataFrame({
        "case_id":[f"C{i:03d}" for i in range(445)],
        "expected":labels
    }).to_csv(v3dir/"SilentGeoBench_v3.csv", index=False)

    pd.DataFrame([
        {"system":"SciGeoGuard_v3","accuracy":1.0,"macro_f1":1.0}
    ]).to_csv(v3dir/"SciGeoGuard_X_v3_metrics.csv", index=False)

    # ----- v6 -----
    v6dir = repo / "results/v6_strong_7b_decomposed_run"
    v6dir.mkdir(parents=True)

    pd.DataFrame([
        {
            "system":"Strong_Qwen_7B",
            "accuracy_vs_generated_reference":0.444444,
            "macro_f1_vs_generated_reference":0.2980,
            "FSCR_vs_generated_reference":0.6964,
            "false_refutation_rate_vs_generated_reference":0.2813
        },
        {
            "system":"Strong_Mistral_7B",
            "accuracy_vs_generated_reference":0.383333,
            "macro_f1_vs_generated_reference":0.3031,
            "FSCR_vs_generated_reference":0.6964,
            "false_refutation_rate_vs_generated_reference":0.1094
        }
    ]).to_csv(v6dir/"Strong_LLM_verdict_metrics.csv", index=False)

    # ----- v8.1 live results -----
    v81 = repo / "results/v8_1_live_corrections"
    v81.mkdir(parents=True)

    regions = [
        "Bengaluru_India",
        "Rotterdam_Netherlands",
        "CentralValley_USA",
        "Manaus_Brazil"
    ]

    # 2 Sentinel replicates/region = 8
    sentinel_rows=[]
    sentinel_vals={
        "Bengaluru_India":[0.069497,0.076910],
        "Rotterdam_Netherlands":[0.110134,0.091553],
        "CentralValley_USA":[0.134453,0.113999],
        "Manaus_Brazil":[0.097577,0.091537]
    }

    for r in regions:
        for j,val in enumerate(sentinel_vals[r]):
            sentinel_rows.append({
                "region":r,
                "date_range":f"fixture_{j+1}",
                "mode":"live",
                "status":"SUCCESS",
                "mean_absolute_index_difference":val,
                "sign_flip_fraction":0.02 + j*0.01
            })

    pd.DataFrame(sentinel_rows).to_csv(
        v81/"Sentinel2_v8_preserved_live_results.csv",
        index=False
    )

    wc_vals={
        "Bengaluru_India":0.145990,
        "Rotterdam_Netherlands":0.190625,
        "CentralValley_USA":0.113906,
        "Manaus_Brazil":0.074863
    }

    pd.DataFrame([
        {
            "region":r,
            "mode":"live",
            "status":"SUCCESS",
            "bilinear_invalid_fraction":wc_vals[r],
            "nearest_valid_fraction":1.0
        }
        for r in regions
    ]).to_csv(
        v81/"WorldCover_v8_preserved_live_results.csv",
        index=False
    )

    dem_p95={
        "Bengaluru_India":1.732312,
        "Rotterdam_Netherlands":1.456681,
        "CentralValley_USA":0.375262,
        "Manaus_Brazil":1.343004
    }

    pd.DataFrame([
        {
            "region":r,
            "mode":"live",
            "status":"SUCCESS",
            "pixel_count_inflation_factor":9.0,
            "derived_output_fraction_without_distinct_source_sample":8/9,
            "roundtrip_core_p95_abs_error_m":dem_p95[r]
        }
        for r in regions
    ]).to_csv(
        v81/"CopDEM_v8_1_corrected_results.csv",
        index=False
    )

    pub_rows = [
        {
            "experiment":"Sentinel-2 spectral-role mismatch",
            "regions":4,
            "live_replicates":8,
            "primary_effect":0.097700,
            "primary_ci95_low":0.073204,
            "primary_ci95_high":0.124226,
            "secondary_effect":0.031936
        },
        {
            "experiment":"WorldCover categorical interpolation",
            "regions":4,
            "live_replicates":4,
            "primary_effect":0.129948,
            "primary_ci95_low":0.074863,
            "primary_ci95_high":0.190625,
            "secondary_effect":1.0
        },
        {
            "experiment":"CopDEM unsupported spatial-resolution gain",
            "regions":4,
            "live_replicates":4,
            "primary_effect":8/9,
            "primary_ci95_low":8/9,
            "primary_ci95_high":8/9,
            "secondary_effect":1.399842
        }
    ]

    pd.DataFrame(pub_rows).to_csv(
        v81/"FINAL_PUBLICATION_RESULTS_TABLE_v8_1.csv",
        index=False
    )

    # Minimal repo metadata/audit targets.
    (repo/"README.md").write_text("# SciGeoGuard-X\n", encoding="utf-8")
    (repo/"CITATION.cff").write_text(
        'cff-version: 1.2.0\ntitle: "SciGeoGuard-X"\ntype: software\n',
        encoding="utf-8"
    )
    (repo/"LICENSE_STATUS.md").write_text(
        "No software license selected yet.\n",
        encoding="utf-8"
    )

    # ----- v9 closure: successful v8.2 -----
    closure.mkdir(parents=True, exist_ok=True)

    dates = {
        "Bengaluru_India":"2024-10-23",
        "Rotterdam_Netherlands":"2024-10-19",
        "CentralValley_USA":"2024-12-23",
        "Manaus_Brazil":"2024-02-14"
    }

    accum = {
        "Bengaluru_India":10.54,
        "Rotterdam_Netherlands":1.65,
        "CentralValley_USA":14.76,
        "Manaus_Brazil":8.88
    }

    pd.DataFrame([
        {
            "region":r,
            "selected_date":dates[r],
            "status":"SUCCESS",
            "grid_cells":9,
            "rainy_cells_ge_1mm":9 if r!="Rotterdam_Netherlands" else 8,
            "mean_daily_accum_mm":accum[r],
            "median_raw_sum_factor":2.0,
            "median_mean_as_accum_fraction":1/24
        }
        for r in regions
    ]).to_csv(
        closure/"FINAL_V8_2_IMERG_RESULTS.csv",
        index=False
    )

    pd.DataFrame([
        {
            "dimension":"v8.2 IMERG multi-region empirical extension",
            "ready":True
        }
    ]).to_csv(
        closure/"FINAL_FCS_READINESS.csv",
        index=False
    )

    (closure/"FINAL_STATUS.json").write_text(
        json.dumps({
            "v8_2_imerg_ready":True,
            "v7_double_human_review_complete":False
        }, indent=2),
        encoding="utf-8"
    )

    repo_zip = INPUTS/"SciGeoGuard-X_GitHub_Ready_All_Code_and_Results.zip"
    closure_zip = INPUTS/"SciGeoGuard_X_v9_Final_Publication_Closure.zip"

    with zipfile.ZipFile(repo_zip,"w",zipfile.ZIP_DEFLATED) as z:
        for p in repo.rglob("*"):
            if p.is_file():
                z.write(p, arcname=str(Path("SciGeoGuard-X")/p.relative_to(repo)))

    with zipfile.ZipFile(closure_zip,"w",zipfile.ZIP_DEFLATED) as z:
        for p in closure.rglob("*"):
            if p.is_file():
                z.write(p, arcname=p.name)

    return repo_zip, closure_zip

if DRY_RUN_FIXTURES:
    repo_zip, closure_zip = make_dryrun_archives()
    print("Created dry-run archives:")
    print(repo_zip)
    print(closure_zip)

## 3. Locate and extract the two required archives

In [ ]:
def runtime_files():
    roots=[ROOT,INPUTS]
    out=[]
    seen=set()

    for r in roots:
        if not r.exists():
            continue

        for p in r.rglob("*"):
            if p.is_file():
                try:
                    key=str(p.resolve())
                except Exception:
                    key=str(p)

                if key not in seen:
                    seen.add(key)
                    out.append(p)

    return out

files_now=runtime_files()

def best_zip(name_contains):
    hits=[
        p for p in files_now
        if p.suffix.lower()==".zip" and
        name_contains.lower() in p.name.lower()
    ]

    if not hits:
        return None

    # Prefer exact/latest largest candidate.
    hits.sort(
        key=lambda p:(
            p.name != name_contains,
            -p.stat().st_size
        )
    )
    return hits[0]

repo_zip = next(
    (
        p for p in files_now
        if p.name=="SciGeoGuard-X_GitHub_Ready_All_Code_and_Results.zip"
    ),
    None
)

closure_zip = next(
    (
        p for p in files_now
        if p.name=="SciGeoGuard_X_v9_Final_Publication_Closure.zip"
    ),
    None
)

if repo_zip is None:
    candidates=[
        p for p in files_now
        if p.suffix.lower()==".zip" and
        "github" in p.name.lower() and
        "scigeoguard" in p.name.lower()
    ]
    repo_zip=candidates[0] if candidates else None

if closure_zip is None:
    candidates=[
        p for p in files_now
        if p.suffix.lower()==".zip" and
        "v9" in p.name.lower() and
        "closure" in p.name.lower()
    ]
    closure_zip=candidates[0] if candidates else None

print("Repository ZIP:",repo_zip)
print("Closure ZIP:",closure_zip)

assert repo_zip is not None, (
    "Upload SciGeoGuard-X_GitHub_Ready_All_Code_and_Results.zip"
)

assert closure_zip is not None, (
    "Upload SciGeoGuard_X_v9_Final_Publication_Closure.zip"
)

In [ ]:
def safe_extract(zpath,dest):
    dest=Path(dest)

    if dest.exists():
        shutil.rmtree(dest)

    dest.mkdir(parents=True)

    with zipfile.ZipFile(zpath) as z:
        for info in z.infolist():
            if info.is_dir():
                continue

            rel=Path(info.filename)

            if rel.is_absolute() or ".." in rel.parts:
                raise RuntimeError(
                    f"Unsafe ZIP entry: {info.filename}"
                )

            out=dest/rel
            out.parent.mkdir(parents=True,exist_ok=True)

            with z.open(info) as fi, open(out,"wb") as fo:
                shutil.copyfileobj(fi,fo)

repo_extract=EXTRACTED/"repository"
closure_extract=EXTRACTED/"closure"

safe_extract(repo_zip,repo_extract)
safe_extract(closure_zip,closure_extract)

print("Extracted repository:",repo_extract)
print("Extracted closure:",closure_extract)

## 4. Canonical file discovery

The selector prefers:

1. canonical `results/` files;
2. live/actual outputs;
3. non-dry-run files.

It rejects fixture markers such as `offline_fixture`.

In [ ]:
ALL_FILES=[
    p for root in [repo_extract,closure_extract]
    for p in root.rglob("*")
    if p.is_file()
]

def fixture_marker(path):
    if path.suffix.lower() not in {
        ".csv",".json",".txt",".md",".ipynb"
    }:
        return False

    try:
        text=path.read_text(
            encoding="utf-8",
            errors="ignore"
        ).lower()
    except Exception:
        return False

    return (
        "offline_fixture" in text or
        "synthetic_fixture" in text
    )

def candidate_score(p):
    s=str(p).lower()
    score=0

    if "/results/" in s or "\\results\\" in s:
        score += 50

    if "live" in s:
        score += 10

    if "dryrun" in s or "dry_run" in s:
        score -= 100

    if "fixture" in s:
        score -= 100

    if fixture_marker(p):
        score -= 200

    return score

def pick(names, required=True):
    if isinstance(names,str):
        names=[names]

    hits=[
        p for p in ALL_FILES
        if p.name in names
    ]

    hits.sort(
        key=lambda p:(
            -candidate_score(p),
            -p.stat().st_size,
            str(p)
        )
    )

    if not hits:
        if required:
            raise FileNotFoundError(
                f"Missing canonical file: {names}"
            )
        return None

    chosen=hits[0]

    if fixture_marker(chosen):
        raise RuntimeError(
            f"Chosen canonical file contains fixture markers: {chosen}"
        )

    return chosen

paths = {
    "v3_benchmark":pick("SilentGeoBench_v3.csv"),
    "v3_metrics":pick("SciGeoGuard_X_v3_metrics.csv",required=False),
    "v6_metrics":pick("Strong_LLM_verdict_metrics.csv"),
    "v6_health":pick("Strong_LLM_health_summary.csv",required=False),

    "sentinel":pick([
        "Sentinel2_v8_preserved_live_results.csv",
        "Sentinel2_multiregion_results.csv"
    ]),

    "worldcover":pick([
        "WorldCover_v8_preserved_live_results.csv",
        "WorldCover_multiregion_results.csv"
    ]),

    "copdem":pick("CopDEM_v8_1_corrected_results.csv"),

    "v81_pub":pick("FINAL_PUBLICATION_RESULTS_TABLE_v8_1.csv"),

    "v82_imerg":pick("FINAL_V8_2_IMERG_RESULTS.csv"),

    "v9_status":pick("FINAL_STATUS.json",required=False),

    "v7_A":pick("Reviewer_A_completed.csv",required=False),
    "v7_B":pick("Reviewer_B_completed.csv",required=False),
    "v7_C":pick("Adjudicator_completed.csv",required=False)
}

for k,p in paths.items():
    print(f"{k:12s} -> {p}")

## 5. Read canonical evidence and validate each layer

In [ ]:
def read_csv(path):
    return pd.read_csv(path)

v3=read_csv(paths["v3_benchmark"])
v6=read_csv(paths["v6_metrics"])
sentinel=read_csv(paths["sentinel"])
worldcover=read_csv(paths["worldcover"])
copdem=read_csv(paths["copdem"])
v81_pub=read_csv(paths["v81_pub"])
imerg=read_csv(paths["v82_imerg"])

expected_col=(
    "expected"
    if "expected" in v3.columns
    else
    "generated_expected"
    if "generated_expected" in v3.columns
    else None
)

assert expected_col is not None, (
    "v3 benchmark has no expected/generated_expected column"
)

v3_counts=v3[expected_col].astype(str).value_counts().to_dict()

print("v3:",len(v3),v3_counts)
display(v6)
display(v81_pub)
display(imerg)

In [ ]:
validation_rows=[]

# ----- v3 -----
v3_ok=(
    len(v3)>0 and
    set(v3[expected_col].astype(str).unique())
    .issubset({"PROVEN","REFUTED","UNPROVEN"})
)

validation_rows.append({
    "gate":"v3 controlled benchmark available",
    "passed":v3_ok,
    "detail":
        f"{len(v3)} cases; {v3_counts}"
})

# ----- v6 -----
required_v6_cols={
    "system",
    "accuracy_vs_generated_reference",
    "FSCR_vs_generated_reference"
}

v6_ok=(
    len(v6)>=2 and
    required_v6_cols.issubset(v6.columns)
)

validation_rows.append({
    "gate":"v6 strong LLM baseline metrics available",
    "passed":v6_ok,
    "detail":
        ", ".join(v6["system"].astype(str))
        if "system" in v6 else ""
})

# ----- Sentinel -----
s_live=sentinel.copy()

if "status" in s_live:
    s_live=s_live[
        s_live.status.astype(str).eq("SUCCESS")
    ]

if "mode" in s_live:
    s_live=s_live[
        s_live["mode"].astype(str).eq("live")
    ]

sentinel_ok=(
    len(s_live)>=8 and
    s_live["region"].nunique()>=4 and
    pd.to_numeric(
        s_live["mean_absolute_index_difference"],
        errors="coerce"
    ).notna().all()
)

validation_rows.append({
    "gate":"Sentinel live multi-region replication",
    "passed":sentinel_ok,
    "detail":
        f"{len(s_live)} successful live replicates; "
        f"{s_live['region'].nunique()} regions"
})

# ----- WorldCover -----
w_live=worldcover.copy()

if "status" in w_live:
    w_live=w_live[
        w_live.status.astype(str).eq("SUCCESS")
    ]

if "mode" in w_live:
    w_live=w_live[
        w_live["mode"].astype(str).eq("live")
    ]

world_ok=(
    len(w_live)>=4 and
    w_live["region"].nunique()>=4 and
    pd.to_numeric(
        w_live["bilinear_invalid_fraction"],
        errors="coerce"
    ).notna().all()
)

validation_rows.append({
    "gate":"WorldCover live multi-region replication",
    "passed":world_ok,
    "detail":
        f"{len(w_live)} live regions"
})

# ----- CopDEM -----
d_live=copdem.copy()

if "status" in d_live:
    d_live=d_live[
        d_live.status.astype(str).eq("SUCCESS")
    ]

if "mode" in d_live:
    d_live=d_live[
        d_live["mode"].astype(str).eq("live")
    ]

copdem_ok=(
    len(d_live)>=4 and
    d_live["region"].nunique()>=4 and
    pd.to_numeric(
        d_live["roundtrip_core_p95_abs_error_m"],
        errors="coerce"
    ).notna().all()
)

validation_rows.append({
    "gate":"CopDEM corrected live multi-region replication",
    "passed":copdem_ok,
    "detail":
        f"{len(d_live)} live regions"
})

# ----- IMERG v8.2 -----
i_live=imerg.copy()

if "status" in i_live:
    i_live=i_live[
        i_live.status.astype(str).eq("SUCCESS")
    ]

imerg_ok=(
    len(i_live)==4 and
    i_live["region"].nunique()==4 and
    "median_raw_sum_factor" in i_live and
    "median_mean_as_accum_fraction" in i_live and
    np.allclose(
        pd.to_numeric(
            i_live.median_raw_sum_factor,
            errors="coerce"
        ),
        2.0,
        rtol=1e-6,
        atol=1e-8
    ) and
    np.allclose(
        pd.to_numeric(
            i_live.median_mean_as_accum_fraction,
            errors="coerce"
        ),
        1/24,
        rtol=1e-6,
        atol=1e-8
    )
)

validation_rows.append({
    "gate":"IMERG v8.2 four-region rainy recovery",
    "passed":imerg_ok,
    "detail":
        (
            f"{len(i_live)} regions; raw factor=2; "
            f"mean fraction=1/24"
        )
})

validation_df=pd.DataFrame(validation_rows)
display(validation_df)

SCIENCE_READY=bool(
    validation_df["passed"].all()
)

print("SCIENCE_READY =",SCIENCE_READY)

## 6. Optional v7 human-review status

Human validation is reported only when genuine completed reviewer files exist.
It is **not** fabricated and it is not required to make the short FCS Code & Data note scientifically coherent, because all controlled-benchmark claims remain explicitly bounded.

In [ ]:
VALID_LABELS={"PROVEN","REFUTED","UNPROVEN"}

def load_completed(path):
    if path is None:
        return pd.DataFrame()

    df=pd.read_csv(path)

    if "reviewer_verdict" not in df.columns:
        return pd.DataFrame()

    df["reviewer_verdict"]=(
        df["reviewer_verdict"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    if not df["reviewer_verdict"].isin(VALID_LABELS).all():
        return pd.DataFrame()

    return df

A=load_completed(paths["v7_A"])
B=load_completed(paths["v7_B"])

V7_DOUBLE_HUMAN=(
    not A.empty and
    not B.empty
)

def cohens_kappa(a,b):
    a=pd.Series(a)
    b=pd.Series(b)

    ok=(
        a.isin(VALID_LABELS) &
        b.isin(VALID_LABELS)
    )

    a=a[ok]
    b=b[ok]

    if len(a)==0:
        return np.nan

    po=float(
        (a.values==b.values).mean()
    )

    pa=a.value_counts(
        normalize=True
    )

    pb=b.value_counts(
        normalize=True
    )

    pe=sum(
        float(pa.get(x,0))*
        float(pb.get(x,0))
        for x in VALID_LABELS
    )

    if math.isclose(1-pe,0):
        return np.nan

    return (po-pe)/(1-pe)

human_summary=pd.DataFrame()

if V7_DOUBLE_HUMAN:
    pair=(
        A[["human_case_id","reviewer_verdict"]]
        .rename(columns={"reviewer_verdict":"reviewer_A"})
        .merge(
            B[["human_case_id","reviewer_verdict"]]
            .rename(columns={"reviewer_verdict":"reviewer_B"}),
            on="human_case_id",
            how="inner"
        )
    )

    human_summary=pd.DataFrame([{
        "n":len(pair),
        "raw_agreement":
            float(
                (pair.reviewer_A==pair.reviewer_B).mean()
            ),
        "cohens_kappa":
            cohens_kappa(
                pair.reviewer_A,
                pair.reviewer_B
            ),
        "disagreements":
            int(
                (pair.reviewer_A!=pair.reviewer_B).sum()
            )
    }])

    display(human_summary)
else:
    print(
        "v7 human review remains pending. "
        "No human-validated accuracy claim will be generated."
    )

## 7. Recompute the final merged result statistics

This produces the numbers that should be used in the manuscript.

In [ ]:
# Sentinel region-level aggregation.
sentinel_region=(
    s_live.groupby("region")
    .agg(
        spectral_mae=(
            "mean_absolute_index_difference",
            "median"
        ),
        sign_flip_fraction=(
            "sign_flip_fraction",
            "median"
        )
    )
    .reset_index()
)

# WorldCover.
world_region=(
    w_live.groupby("region")
    .agg(
        invalid_fraction=(
            "bilinear_invalid_fraction",
            "median"
        ),
        nearest_valid_fraction=(
            "nearest_valid_fraction",
            "median"
        )
    )
    .reset_index()
)

# CopDEM.
dem_region=(
    d_live.groupby("region")
    .agg(
        p95_error_m=(
            "roundtrip_core_p95_abs_error_m",
            "median"
        ),
        derived_fraction=(
            "derived_output_fraction_without_distinct_source_sample",
            "median"
        ),
        pixel_inflation=(
            "pixel_count_inflation_factor",
            "median"
        )
    )
    .reset_index()
)

# IMERG.
imerg_region=i_live.copy()

# Use the corrected v8.1 publication table when available for the
# bootstrap-derived Sentinel/WorldCover confidence intervals.
def pubrow(fragment):
    m=v81_pub[
        v81_pub["experiment"]
        .astype(str)
        .str.contains(
            fragment,
            case=False,
            regex=False
        )
    ]

    return (
        m.iloc[0]
        if len(m)
        else None
    )

sent_pub=pubrow("Sentinel")
wc_pub=pubrow("WorldCover")
dem_pub=pubrow("CopDEM")

merged_summary = {
    "v3_cases":int(len(v3)),
    "v3_proven":int(v3_counts.get("PROVEN",0)),
    "v3_refuted":int(v3_counts.get("REFUTED",0)),
    "v3_unproven":int(v3_counts.get("UNPROVEN",0)),

    "sentinel_region_median_mae":
        float(sentinel_region.spectral_mae.median()),

    "sentinel_region_median_sign_flip":
        float(sentinel_region.sign_flip_fraction.median()),

    "worldcover_region_median_invalid_fraction":
        float(world_region.invalid_fraction.median()),

    "worldcover_nearest_valid_fraction":
        float(world_region.nearest_valid_fraction.median()),

    "copdem_region_median_p95_error_m":
        float(dem_region.p95_error_m.median()),

    "copdem_derived_fraction":
        float(dem_region.derived_fraction.median()),

    "copdem_pixel_inflation":
        float(dem_region.pixel_inflation.median()),

    "imerg_regions":
        int(imerg_region.region.nunique()),

    "imerg_raw_sum_factor":
        float(
            pd.to_numeric(
                imerg_region.median_raw_sum_factor,
                errors="coerce"
            ).median()
        ),

    "imerg_mean_fraction":
        float(
            pd.to_numeric(
                imerg_region.median_mean_as_accum_fraction,
                errors="coerce"
            ).median()
        )
}

if sent_pub is not None:
    merged_summary["sentinel_ci_low"]=float(sent_pub.get("primary_ci95_low",np.nan))
    merged_summary["sentinel_ci_high"]=float(sent_pub.get("primary_ci95_high",np.nan))

if wc_pub is not None:
    merged_summary["worldcover_ci_low"]=float(wc_pub.get("primary_ci95_low",np.nan))
    merged_summary["worldcover_ci_high"]=float(wc_pub.get("primary_ci95_high",np.nan))

print(json.dumps(merged_summary,indent=2))

## 8. Final evidence table

The table distinguishes controlled benchmark evidence, model baselines, live counterfactual evidence, and pending human validation.

In [ ]:
evidence_rows=[]

evidence_rows.append({
    "layer":"Controlled benchmark",
    "version":"v3",
    "status":"COMPLETE",
    "result":
        f"{merged_summary['v3_cases']} cases: "
        f"{merged_summary['v3_proven']} PROVEN, "
        f"{merged_summary['v3_refuted']} REFUTED, "
        f"{merged_summary['v3_unproven']} UNPROVEN",
    "boundary":
        "Generated scientific-invariant reference; not independent human ground truth"
})

for _,r in v6.iterrows():
    evidence_rows.append({
        "layer":"Strong decomposed LLM baseline",
        "version":"v6",
        "status":"COMPLETE",
        "result":
            f"{r['system']}: "
            f"accuracy={float(r['accuracy_vs_generated_reference']):.4f}; "
            f"FSCR={float(r['FSCR_vs_generated_reference']):.4f}",
        "boundary":
            "Baseline against generated reference; not a scientific adjudicator"
    })

evidence_rows.extend([
    {
        "layer":"Live EO counterfactual",
        "version":"v8.1",
        "status":"COMPLETE",
        "result":
            f"Sentinel-2 median regional MAE="
            f"{merged_summary['sentinel_region_median_mae']:.4f}",
        "boundary":
            "Real-input spectral-role stress test"
    },
    {
        "layer":"Live EO counterfactual",
        "version":"v8.1",
        "status":"COMPLETE",
        "result":
            f"WorldCover median invalid bilinear class fraction="
            f"{merged_summary['worldcover_region_median_invalid_fraction']:.4f}",
        "boundary":
            "Real-input categorical-resampling stress test"
    },
    {
        "layer":"Live EO counterfactual",
        "version":"v8.1",
        "status":"COMPLETE",
        "result":
            f"CopDEM 3x/axis -> "
            f"{merged_summary['copdem_pixel_inflation']:.1f}x cells; "
            f"derived fraction="
            f"{merged_summary['copdem_derived_fraction']:.4f}; "
            f"median regional p95 round-trip error="
            f"{merged_summary['copdem_region_median_p95_error_m']:.4f} m",
        "boundary":
            "Real-input unsupported-resolution-claim stress test"
    },
    {
        "layer":"Live EO counterfactual",
        "version":"v8.2/v9",
        "status":"COMPLETE",
        "result":
            f"IMERG rainy recovery in {merged_summary['imerg_regions']}/4 regions; "
            f"raw-sum factor={merged_summary['imerg_raw_sum_factor']:.1f}x; "
            f"mean-rate fraction={merged_summary['imerg_mean_fraction']:.6f}",
        "boundary":
            "Real-input temporal quantity-semantics stress test"
    },
    {
        "layer":"Independent blinded human review",
        "version":"v7",
        "status":
            "COMPLETE"
            if V7_DOUBLE_HUMAN
            else "PENDING",
        "result":
            (
                f"agreement={human_summary.iloc[0].raw_agreement:.4f}; "
                f"kappa={human_summary.iloc[0].cohens_kappa:.4f}"
            )
            if V7_DOUBLE_HUMAN
            else
            "Reviewer package exists; completed A/B verdicts not present",
        "boundary":
            (
                "Independent human agreement available"
                if V7_DOUBLE_HUMAN
                else
                "Do not claim human-validated system accuracy"
            )
    }
])

final_evidence=pd.DataFrame(evidence_rows)
display(final_evidence)

## 9. Generate the real final FCS composite figure

This figure contains only actual merged evidence:

- **A** Sentinel-2 spectral-role mismatch;
- **B** WorldCover invalid class fraction;
- **C** corrected CopDEM p95 round-trip error;
- **D** IMERG four-region duration-semantics result.

No "missing result" panels remain.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

FIGDIR=OUTPUTS/"figure_panels"
FIGDIR.mkdir(exist_ok=True)

pretty={
    "Bengaluru_India":"Bengaluru",
    "Rotterdam_Netherlands":"Rotterdam",
    "CentralValley_USA":"Central Valley",
    "Manaus_Brazil":"Manaus"
}

order=[
    "Bengaluru_India",
    "Rotterdam_Netherlands",
    "CentralValley_USA",
    "Manaus_Brazil"
]

panels=[]

# A Sentinel
p=FIGDIR/"A_sentinel.png"
x=sentinel_region.set_index("region").reindex(order).reset_index()

fig,ax=plt.subplots(figsize=(7,4.5))
ax.bar(
    [pretty.get(r,r) for r in x.region],
    x.spectral_mae
)
ax.set_ylabel("Regional median mean absolute index difference")
ax.set_title("Sentinel-2 spectral-role mismatch")
ax.tick_params(axis="x",rotation=20)
fig.tight_layout()
fig.savefig(p,dpi=300,bbox_inches="tight")
plt.close(fig)
panels.append(p)

# B WorldCover
p=FIGDIR/"B_worldcover.png"
x=world_region.set_index("region").reindex(order).reset_index()

fig,ax=plt.subplots(figsize=(7,4.5))
ax.bar(
    [pretty.get(r,r) for r in x.region],
    x.invalid_fraction
)
ax.set_ylabel("Bilinear invalid class-code fraction")
ax.set_title("ESA WorldCover categorical interpolation")
ax.tick_params(axis="x",rotation=20)
fig.tight_layout()
fig.savefig(p,dpi=300,bbox_inches="tight")
plt.close(fig)
panels.append(p)

# C CopDEM
p=FIGDIR/"C_copdem.png"
x=dem_region.set_index("region").reindex(order).reset_index()

fig,ax=plt.subplots(figsize=(7,4.5))
ax.bar(
    [pretty.get(r,r) for r in x.region],
    x.p95_error_m
)
ax.set_ylabel("Aligned core p95 round-trip error (m)")
ax.set_title("Copernicus DEM aligned resampling")
ax.tick_params(axis="x",rotation=20)
fig.tight_layout()
fig.savefig(p,dpi=300,bbox_inches="tight")
plt.close(fig)
panels.append(p)

# D IMERG
p=FIGDIR/"D_imerg.png"
x=imerg_region.set_index("region").reindex(order).reset_index()

fig,ax=plt.subplots(figsize=(7,4.5))
ax.bar(
    [pretty.get(r,r) for r in x.region],
    x.median_raw_sum_factor
)
ax.axhline(2.0,linestyle="--",linewidth=1)
ax.set_ylim(0,2.4)
ax.set_ylabel("Raw rate-sum / correct accumulation")
ax.set_title("GPM IMERG temporal quantity semantics")
ax.tick_params(axis="x",rotation=20)
fig.tight_layout()
fig.savefig(p,dpi=300,bbox_inches="tight")
plt.close(fig)
panels.append(p)

imgs=[Image.open(p).convert("RGB") for p in panels]
w=max(im.width for im in imgs)
h=max(im.height for im in imgs)

prepared=[]

for idx,im in enumerate(imgs):
    canvas=Image.new(
        "RGB",
        (w,h),
        "white"
    )

    canvas.paste(
        im,
        (
            (w-im.width)//2,
            (h-im.height)//2
        )
    )

    ImageDraw.Draw(canvas).text(
        (18,14),
        chr(ord("A")+idx),
        fill="black"
    )

    prepared.append(canvas)

final=Image.new(
    "RGB",
    (2*w,2*h),
    "white"
)

final.paste(prepared[0],(0,0))
final.paste(prepared[1],(w,0))
final.paste(prepared[2],(0,h))
final.paste(prepared[3],(w,h))

FINAL_FIG=OUTPUTS/"FINAL_FCS_COMPOSITE_FIGURE_v9_1.png"
final.save(
    FINAL_FIG,
    dpi=(300,300)
)

print(FINAL_FIG)

## 10. Generate manuscript-ready FCS result text

The paragraph is written to stay within the evidence boundaries.

It may be pasted into the short FCS Letter.

In [ ]:
qwen = v6[
    v6["system"].astype(str)
    .str.contains("Qwen",case=False)
]

mistral = v6[
    v6["system"].astype(str)
    .str.contains("Mistral",case=False)
]

qwen_acc = (
    float(qwen.iloc[0]["accuracy_vs_generated_reference"])
    if len(qwen) else np.nan
)

qwen_fscr = (
    float(qwen.iloc[0]["FSCR_vs_generated_reference"])
    if len(qwen) else np.nan
)

mistral_acc = (
    float(mistral.iloc[0]["accuracy_vs_generated_reference"])
    if len(mistral) else np.nan
)

mistral_fscr = (
    float(mistral.iloc[0]["FSCR_vs_generated_reference"])
    if len(mistral) else np.nan
)

sent_ci = ""

if (
    "sentinel_ci_low" in merged_summary and
    np.isfinite(merged_summary["sentinel_ci_low"])
):
    sent_ci = (
        f" (95% region-bootstrap CI "
        f"{merged_summary['sentinel_ci_low']:.4f}–"
        f"{merged_summary['sentinel_ci_high']:.4f})"
    )

wc_ci = ""

if (
    "worldcover_ci_low" in merged_summary and
    np.isfinite(merged_summary["worldcover_ci_low"])
):
    wc_ci = (
        f" (95% CI "
        f"{100*merged_summary['worldcover_ci_low']:.2f}%–"
        f"{100*merged_summary['worldcover_ci_high']:.2f}%)"
    )

paragraph = f"""
A frozen SciGeoGuard-X verifier was evaluated on {merged_summary['v3_cases']} controlled
SilentGeoBench workflows ({merged_summary['v3_proven']} PROVEN,
{merged_summary['v3_refuted']} REFUTED, and {merged_summary['v3_unproven']} UNPROVEN).
These generated reference labels test consistency with encoded scientific invariants and are
not independent human ground truth. Against the same generated reference, the decomposed
Qwen2.5-7B baseline achieved {qwen_acc:.2%} accuracy with an FSCR of {qwen_fscr:.2%},
while Mistral-7B achieved {mistral_acc:.2%} accuracy with an FSCR of {mistral_fscr:.2%}.

Live-data counterfactuals were then evaluated in Bengaluru, Rotterdam, California's Central
Valley, and Manaus. Replacing Sentinel-2 B08 NIR with B11 SWIR in an NDVI-labelled
calculation produced a regional median mean absolute index difference of
{merged_summary['sentinel_region_median_mae']:.4f}{sent_ci}. Bilinear interpolation of
ESA WorldCover nominal classes produced a median
{100*merged_summary['worldcover_region_median_invalid_fraction']:.2f}% invalid class values
across regions{wc_ci}, whereas nearest-neighbour resampling preserved valid class codes in
{100*merged_summary['worldcover_nearest_valid_fraction']:.0f}% of evaluated outputs.
For Copernicus DEM GLO-30, threefold interpolation along each raster axis generated
{merged_summary['copdem_pixel_inflation']:.0f} times as many grid cells, so
{100*merged_summary['copdem_derived_fraction']:.2f}% of fine-grid positions did not correspond
to distinct source samples; the corrected region-median core p95 native-to-fine-to-native
round-trip error was {merged_summary['copdem_region_median_p95_error_m']:.4f} m.
Finally, live rainy GPM IMERG V07 cases were recovered in all
{merged_summary['imerg_regions']} study regions. Omitting the 0.5 h duration when integrating
half-hourly precipitation rates yielded the expected
{merged_summary['imerg_raw_sum_factor']:.1f}× numeric overstatement, while treating the
full-day mean rate as daily accumulation yielded
{merged_summary['imerg_mean_fraction']:.6f} (=1/24) of the correct accumulation.
""".strip()

print(paragraph)

(OUTPUTS/"FINAL_FCS_RESULTS_PARAGRAPH.txt").write_text(
    paragraph,
    encoding="utf-8"
)

latex_paragraph = (
    paragraph
    .replace("%","\\%")
    .replace("×","$\\times$")
    .replace("–","--")
)

(OUTPUTS/"FINAL_FCS_RESULTS_PARAGRAPH.tex").write_text(
    latex_paragraph,
    encoding="utf-8"
)

## 11. Final figure caption

In [ ]:
caption = (
    "Final merged live-data counterfactuals across four regions. "
    "(a) Sentinel-2 spectral-role mismatch, reported as the regional median mean absolute "
    "difference between the correct B08/B04 NDVI and the B11-substitution variant. "
    "(b) Fraction of invalid ESA WorldCover nominal class codes produced by bilinear interpolation. "
    "(c) Corrected Copernicus DEM native-to-fine-to-native core p95 interpolation error using "
    "geospatially aligned reprojection; threefold interpolation per axis creates nine times as many "
    "grid cells without additional source observations. "
    "(d) GPM IMERG V07 precipitation-rate semantics: summing half-hourly mm/h values without the "
    "required 0.5 h duration produces a 2× numeric overstatement in each recovered rainy region."
)

print(caption)

(OUTPUTS/"FINAL_FCS_FIGURE_CAPTION.txt").write_text(
    caption,
    encoding="utf-8"
)

## 12. Audit the public-release repository

Science readiness and release readiness are separate.

A missing software license or manuscript placeholder does **not** invalidate the experiments,
but it blocks calling the GitHub repository a final public/open-source release.

In [ ]:
# Find the repository root inside extracted repository ZIP.
repo_roots=[]

for p in repo_extract.rglob("README.md"):
    parent=p.parent

    if (
        (parent/"notebooks").exists() and
        (parent/"results").exists()
    ):
        repo_roots.append(parent)

REPO_ROOT=(
    sorted(
        repo_roots,
        key=lambda p:len(str(p))
    )[0]
    if repo_roots
    else None
)

print("REPO_ROOT =",REPO_ROOT)

audit=[]

if REPO_ROOT is not None:
    repo_files=[
        p for p in REPO_ROOT.rglob("*")
        if p.is_file()
    ]

    has_license=any(
        p.name.lower() in {
            "license",
            "license.txt",
            "license.md",
            "copying"
        }
        for p in repo_files
    )

    placeholder_terms=[
        "REPLACE-BEFORE-SUBMISSION",
        "REPLACE-WITH-CORRESPONDING-EMAIL",
        "[GITHUB-URL]",
        "[ZENODO-DOI]",
        "[corresponding-author-email]"
    ]

    placeholder_hits=[]

    for p in repo_files:
        if p.suffix.lower() not in {
            ".md",".txt",".tex",".json",".cff",".ipynb",".py",".yaml",".yml"
        }:
            continue

        try:
            text=p.read_text(
                encoding="utf-8",
                errors="ignore"
            )
        except Exception:
            continue

        for term in placeholder_terms:
            if term in text:
                placeholder_hits.append(
                    str(p.relative_to(REPO_ROOT))
                    + " :: " + term
                )

    oversized=[
        str(p.relative_to(REPO_ROOT))
        for p in repo_files
        if p.stat().st_size > 100*1024*1024
    ]

    secret_patterns={
        "OpenAI":re.compile(r"sk-[A-Za-z0-9_-]{20,}"),
        "GitHub":re.compile(r"gh[pousr]_[A-Za-z0-9]{20,}"),
        "AWS":re.compile(r"AKIA[0-9A-Z]{16}"),
        "HuggingFace":re.compile(r"hf_[A-Za-z0-9]{20,}")
    }

    secrets=[]

    for p in repo_files:
        if p.suffix.lower() not in {
            ".md",".txt",".tex",".json",".cff",".ipynb",".py",".yaml",".yml"
        }:
            continue

        try:
            text=p.read_text(
                encoding="utf-8",
                errors="ignore"
            )
        except Exception:
            continue

        for name,pat in secret_patterns.items():
            if pat.search(text):
                secrets.append(
                    str(p.relative_to(REPO_ROOT))
                    + " :: " + name
                )

    audit.extend([
        {
            "gate":"README present",
            "passed":(REPO_ROOT/"README.md").exists(),
            "detail":""
        },
        {
            "gate":"CITATION.cff present",
            "passed":(REPO_ROOT/"CITATION.cff").exists(),
            "detail":""
        },
        {
            "gate":"Explicit software license present",
            "passed":has_license,
            "detail":
                "LICENSE_STATUS.md is not a license"
        },
        {
            "gate":"No >100 MB files",
            "passed":len(oversized)==0,
            "detail":"; ".join(oversized[:20])
        },
        {
            "gate":"No likely secrets",
            "passed":len(secrets)==0,
            "detail":"; ".join(secrets[:20])
        },
        {
            "gate":"No submission placeholders",
            "passed":len(placeholder_hits)==0,
            "detail":"; ".join(placeholder_hits[:30])
        }
    ])

release_audit=pd.DataFrame(audit)
display(release_audit)

RELEASE_READY=(
    bool(release_audit["passed"].all())
    if len(release_audit)
    else False
)

print("RELEASE_READY =",RELEASE_READY)

## 13. Final readiness

The core scientific evidence should now be fully ready.

The only allowed remaining scientific item is optional v7 human review.
Release blockers are administrative.

In [ ]:
readiness=pd.DataFrame([
    {
        "dimension":"Core scientific evidence",
        "ready":SCIENCE_READY,
        "blocking_submission":True,
        "detail":
            "v3 + v6 + Sentinel + WorldCover + CopDEM + successful 4-region IMERG"
    },
    {
        "dimension":"Independent v7 human validation",
        "ready":V7_DOUBLE_HUMAN,
        "blocking_submission":False,
        "detail":
            "Optional strengthening; do not claim human-validated accuracy while pending"
    },
    {
        "dimension":"GitHub/public release",
        "ready":RELEASE_READY,
        "blocking_submission":True,
        "detail":
            "Administrative: license + placeholders + repository hygiene"
    }
])

display(readiness)

FINAL_SCIENTIFIC_DECISION = (
    "READY"
    if SCIENCE_READY
    else "NOT READY"
)

print(
    "FINAL_SCIENTIFIC_DECISION =",
    FINAL_SCIENTIFIC_DECISION
)

## 14. Freeze the canonical evidence files

The final package contains copies of only the canonical files used in the final result.
This gives reviewers and future readers a small, auditable evidence set.

In [ ]:
canonical_copy = {
    "v3/SilentGeoBench_v3.csv":
        paths["v3_benchmark"],

    "v3/SciGeoGuard_X_v3_metrics.csv":
        paths["v3_metrics"],

    "v6/Strong_LLM_verdict_metrics.csv":
        paths["v6_metrics"],

    "v6/Strong_LLM_health_summary.csv":
        paths["v6_health"],

    "v8_1/Sentinel2_live_results.csv":
        paths["sentinel"],

    "v8_1/WorldCover_live_results.csv":
        paths["worldcover"],

    "v8_1/CopDEM_corrected_results.csv":
        paths["copdem"],

    "v8_1/FINAL_PUBLICATION_RESULTS_TABLE_v8_1.csv":
        paths["v81_pub"],

    "v8_2/FINAL_V8_2_IMERG_RESULTS.csv":
        paths["v82_imerg"]
}

for rel,src in canonical_copy.items():
    if src is None:
        continue

    dst=FROZEN/rel
    dst.parent.mkdir(parents=True,exist_ok=True)
    shutil.copy2(src,dst)

if V7_DOUBLE_HUMAN:
    for rel,src in [
        ("v7/Reviewer_A_completed.csv",paths["v7_A"]),
        ("v7/Reviewer_B_completed.csv",paths["v7_B"])
    ]:
        dst=FROZEN/rel
        dst.parent.mkdir(parents=True,exist_ok=True)
        shutil.copy2(src,dst)

print("Frozen canonical files:")
for p in sorted(FROZEN.rglob("*")):
    if p.is_file():
        print(" -",p.relative_to(FROZEN))

## 15. Export the final merge package

In [ ]:
final_evidence.to_csv(
    OUTPUTS/"FINAL_EVIDENCE_TABLE_v9_1.csv",
    index=False
)

validation_df.to_csv(
    OUTPUTS/"FINAL_SCIENCE_GATES_v9_1.csv",
    index=False
)

readiness.to_csv(
    OUTPUTS/"FINAL_READINESS_v9_1.csv",
    index=False
)

release_audit.to_csv(
    OUTPUTS/"FINAL_RELEASE_AUDIT_v9_1.csv",
    index=False
)

sentinel_region.to_csv(
    OUTPUTS/"FINAL_SENTINEL_REGION_RESULTS.csv",
    index=False
)

world_region.to_csv(
    OUTPUTS/"FINAL_WORLDCOVER_REGION_RESULTS.csv",
    index=False
)

dem_region.to_csv(
    OUTPUTS/"FINAL_COPDEM_REGION_RESULTS.csv",
    index=False
)

imerg_region.to_csv(
    OUTPUTS/"FINAL_IMERG_REGION_RESULTS.csv",
    index=False
)

if not human_summary.empty:
    human_summary.to_csv(
        OUTPUTS/"FINAL_V7_HUMAN_AGREEMENT.csv",
        index=False
    )

(OUTPUTS/"FINAL_MERGED_NUMBERS.json").write_text(
    json.dumps(
        merged_summary,
        indent=2
    ),
    encoding="utf-8"
)

status = {
    "created_utc":
        datetime.now(timezone.utc).isoformat(),

    "science_ready":
        bool(SCIENCE_READY),

    "v7_human_complete":
        bool(V7_DOUBLE_HUMAN),

    "release_ready":
        bool(RELEASE_READY),

    "final_scientific_decision":
        FINAL_SCIENTIFIC_DECISION,

    "paper_policy":
        (
            "Use v3 as controlled reference evidence; "
            "v6 as LLM baselines; v8.1 and v8.2 as real-input counterfactual replication. "
            "Do not claim human-validated accuracy unless v7 is complete."
        )
}

(OUTPUTS/"FINAL_STATUS_v9_1.json").write_text(
    json.dumps(
        status,
        indent=2
    ),
    encoding="utf-8"
)

# Hash the entire output tree except the hash file itself.
hash_rows=[]

for p in sorted(OUTPUTS.rglob("*")):
    if not p.is_file():
        continue

    if p.name=="FINAL_FILE_HASHES_SHA256_v9_1.csv":
        continue

    hash_rows.append({
        "path":
            str(p.relative_to(OUTPUTS)),
        "bytes":
            p.stat().st_size,
        "sha256":
            hashlib.sha256(
                p.read_bytes()
            ).hexdigest()
    })

pd.DataFrame(hash_rows).to_csv(
    OUTPUTS/"FINAL_FILE_HASHES_SHA256_v9_1.csv",
    index=False
)

zip_path=shutil.make_archive(
    str(
        ROOT/
        "SciGeoGuard_X_v9_1_Final_Merge_and_Freeze"
    ),
    "zip",
    root_dir=OUTPUTS
)

print("Created:",zip_path)
print(json.dumps(status,indent=2))

try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    pass

# Final rule

When this notebook prints:

```text
FINAL_SCIENTIFIC_DECISION = READY
```

the experimental work for the FCS short Code & Data note is finished.

Do **not** create v10 for another benchmark/model experiment.

After that, only:

1. choose/add the software license;
2. replace public-release/manuscript placeholders;
3. publish the GitHub release;
4. optionally archive a DOI;
5. replace the paper's result paragraph and figure with the v9.1 exports;
6. submit under **Code & Data in Earth Science**.

v7 human review can still be completed as optional strengthening or for a later full paper.